# Dataset de modelado y separación train/test

Arma el dataset único para la Etapa A de la consigna (`consigna/Consigna TP3.pdf`): une las features de
`3_analisis_temporal.ipynb` (lags, rolling, encoding cíclico), los pronósticos y errores de
`2_relaciones-bivariadas.ipynb` y los atributos geográficos de `4_atributos_geograficos.ipynb` en una sola tabla, define
qué filas tienen un target válido para cada variable y separa train/test por fecha.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import AZUL, NARANJA, GRIS_TEXTO, aplicar_estilo

In [ ]:
PROCESSED = '../data/processed/'
OUT = '../data/out/modelado/'

# Paleta, colormaps y funciones de gráficos: notebooks/utils.py
aplicar_estilo()

figuras = {}  # se exportan todas al final

## 1. Carga de los tres archivos a unir

In [ ]:
panel = pd.read_csv(PROCESSED + 'panel_features.csv')
panel['target_hour_utc'] = pd.to_datetime(panel['target_hour_utc'], utc=True)

mediciones = pd.read_csv(PROCESSED + 'mediciones_limpias.csv')
mediciones['target_hour_utc'] = pd.to_datetime(mediciones['target_hour_utc'], utc=True)

atributos = pd.read_csv(PROCESSED + 'estaciones_atributos.csv')

print(f'panel_features:        {panel.shape[0]:>7,} filas, {panel.shape[1]:>2} columnas, {panel.station_id.nunique()} estaciones')
print(f'mediciones_limpias:    {mediciones.shape[0]:>7,} filas, {mediciones.shape[1]:>2} columnas, {mediciones.station_id.nunique()} estaciones')
print(f'estaciones_atributos:  {atributos.shape[0]:>7,} filas, {atributos.shape[1]:>2} columnas')

`panel_features.csv` trae sólo las 324 estaciones del cohorte con intermitencia manejable (sección 4 de
`3_analisis_temporal.ipynb`), así que es el ancla de la unión: `mediciones_limpias.csv` y `estaciones_atributos.csv`
tienen las 498 estaciones del proyecto completo y se agregan con `how='left'` sobre ese cohorte.

## 2. Unión en un único dataset de modelado

Clave de unión: `station_id` + `target_hour_utc`. Es una clave simple porque `horizonte_horas` es constante (6 h en
todo el dataset: no hay pronósticos emitidos a distintos horizontes para la misma hora objetivo que compitan en la
unión). De `mediciones_limpias.csv` se traen sólo los pronósticos (`fcst_*`) y los errores ya calculados en la sección 1
de `relaciones-bivariadas.ipynb` (`error_<var>` = pronosticado − observado, y su valor absoluto), no las columnas
observadas: esas ya están en `panel_features.csv`, con lags y flags de imputación que `mediciones_limpias.csv` no
tiene.

In [ ]:
VARIABLES_TARGET = ['temp_c', 'rh_pct', 'dewpoint_c', 'windspeed_kmh']
COLUMNAS_FCST = [f'fcst_{v}' for v in VARIABLES_TARGET] + ['fcst_pop_pct']
COLUMNAS_ERROR = [f'error_{v}' for v in VARIABLES_TARGET] + [f'abs_error_{v}' for v in VARIABLES_TARGET]

assert mediciones['horizonte_horas'].nunique() == 1, 'hay más de un horizonte de pronóstico: la unión por estación+hora sería ambigua'
assert not mediciones.duplicated(['station_id', 'target_hour_utc']).any(), 'mediciones_limpias tiene pares estación+hora repetidos'
assert not panel.duplicated(['station_id', 'target_hour_utc']).any(), 'panel_features tiene pares estación+hora repetidos'

dataset = panel.merge(
    mediciones[['station_id', 'target_hour_utc'] + COLUMNAS_FCST + COLUMNAS_ERROR],
    on=['station_id', 'target_hour_utc'], how='left')

dataset = dataset.merge(atributos, on='station_id', how='left')

assert dataset['bioma'].notna().all(), 'alguna estación del cohorte quedó sin atributos geográficos'
print(f'dataset: {dataset.shape[0]:,} filas, {dataset.shape[1]} columnas, {dataset.station_id.nunique()} estaciones')

## 3. Validez del target: no toda fila tiene un error observable

`panel_features.csv` reindexa cada estación del cohorte a la grilla horaria completa del proyecto e imputa huecos
cortos (sección 8 de `3_analisis_temporal.ipynb`); `mediciones_limpias.csv` sólo tiene las filas con una medición real.
Al traer el error con `how='left'`, las horas que en el panel son puramente imputadas (no hubo observación real) o que
`relaciones-bivariadas.ipynb` descartó por inválida (diferencia mayor a 15 °C o 120 km/h contra el pronóstico) quedan en
`NaN`: no hay observación genuina con la que comparar el pronóstico, así que no pueden ser el target del modelo de
corrección.

Se marca la validez por variable (cada una tiene su propia tasa de imputación) en vez de exigir las 4 a la vez, para no
descartar filas de más: una fila puede tener temperatura válida y viento imputado.

In [ ]:
for v in VARIABLES_TARGET:
    dataset[f'target_valido_{v}'] = dataset[f'error_{v}'].notna()

filas = []
for v in VARIABLES_TARGET:
    valido = dataset[f'target_valido_{v}']
    imputado = dataset[f'obs_{v}_imputado']
    filas.append({
        'variable': v,
        'target_valido': int(valido.sum()),
        'target_valido_pct': round(100 * valido.mean(), 1),
        'invalido_hora_imputada': int((~valido & imputado).sum()),
        'invalido_otro_motivo': int((~valido & ~imputado).sum()),
    })

resumen_validez = pd.DataFrame(filas).set_index('variable')
resumen_validez

Las 4 variables rondan el 79 % de filas con target válido. De las inválidas, la gran mayoría (~93-95 %) son horas
imputadas -el motivo esperado-; el resto (~1,4-1,7 % del total) son lecturas que `relaciones-bivariadas.ipynb` ya había
descartado por inválidas. Ninguna fila se elimina acá: cada notebook de modelado filtra por `target_valido_<variable>`
según la variable que esté entrenando.

## 4. Separación train/test

No corresponde un split aleatorio ni K-fold estándar: `3_analisis_temporal.ipynb` confirmó autocorrelación horaria
alta (lags 1-3) y un ciclo diario significativo en todo el cohorte, así que una fila de test cercana en el tiempo a una
fila de train estaría filtrada por información casi idéntica -el modelo no estaría prediciendo, estaría memorizando el
entorno inmediato. Tampoco hace falta un *gap* entre train y test: todas las features (lags, rolling, vecinas) miran
sólo hacia atrás (`shift(1)` o más, sección "Feature engineering" de `3_analisis_temporal.ipynb`), así que una fila de
test al principio del período de test sólo usa información de horas anteriores, reales o de train.

El corte es por fecha, igual para las 324 estaciones (no por estación): el dataset tiene 15 días (19/10 al 2/11), con un
volumen de filas por día casi constante (~7.776, salvo los dos días parciales de los extremos). Se separa el último
~20 % como test -el 31/10 en adelante-, dejando ~12 días para entrenar.

In [ ]:
CORTE = pd.Timestamp('2026-10-31', tz='UTC')

dataset['split'] = np.where(dataset['target_hour_utc'] < CORTE, 'train', 'test')

resumen_split = dataset.groupby('split').agg(
    filas=('station_id', 'size'),
    estaciones=('station_id', 'nunique'),
    desde=('target_hour_utc', 'min'),
    hasta=('target_hour_utc', 'max'),
)
resumen_split['filas_pct'] = (100 * resumen_split['filas'] / len(dataset)).round(1)
resumen_split

In [ ]:
# Las 324 estaciones del cohorte aparecen en los dos splits, y la tasa de target válido no se degrada en test
# (si un split perdiera casi toda su cobertura de alguna variable, el corte elegido no serviría).
resumen_validez_split = pd.concat({
    split: pd.DataFrame({
        v: {'filas': len(g), 'target_valido_pct': round(100 * g[f'target_valido_{v}'].mean(), 1)}
        for v in VARIABLES_TARGET
    }).T
    for split, g in dataset.groupby('split')
}, axis=1)
resumen_validez_split

Chequeo visual: filas por día, coloreadas por split.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
conteo_dia = dataset.groupby([dataset['target_hour_utc'].dt.date, 'split']).size().unstack('split').fillna(0)
conteo_dia[['train', 'test']].plot(kind='bar', stacked=True, ax=ax, color=[AZUL, NARANJA], width=0.85)
ax.set_title('Filas por día, train vs. test')
ax.set_xlabel('')
ax.set_ylabel('Filas')
ax.legend(frameon=False)
fig.tight_layout()
figuras['01_split_por_dia'] = fig

## 5. Guardado

Se guarda el dataset completo (features + pronósticos + errores + atributos geográficos + flags de validez +
columna `split`) en una sola tabla: los notebooks de modelado filtran por `split` y por `target_valido_<variable>`
según la variable que entrenen, en vez de depender de archivos train/test separados que haya que mantener
sincronizados.

In [ ]:
import os

dataset.to_csv(PROCESSED + 'dataset_modelado.csv', index=False)
print(f'{PROCESSED}dataset_modelado.csv: {dataset.shape[0]:,} filas, {dataset.shape[1]} columnas')

os.makedirs(OUT, exist_ok=True)
for nombre, fig in figuras.items():
    fig.savefig(OUT + f'{nombre}.png', dpi=300, bbox_inches='tight', facecolor='white')
resumen_validez.to_csv(OUT + 'tabla_validez_target.csv')
resumen_split.to_csv(OUT + 'tabla_resumen_split.csv')
print(f'Guardados {len(figuras)} gráfico(s) y 2 tablas en {OUT}')